# Data Preprocessing

###Load Packages

In [ ]:
!pip install langchain pypdf sentence-transformers ctransformers chromadb -q
!pip install --upgrade langchain-community
!mkdir docs
!pip install wget -q
!pip install rank_bm25

ERROR: Operation cancelled by user
mkdir: cannot create directory ‘docs’: File exists


###Importing Documents

In [ ]:
import wget
import os
import requests

def get_github_files(repo_owner, repo_name, directory_path=""):
    """
    Fetches a list of PDF files from a GitHub repository directory.

    Args:
        repo_owner (str): The owner of the GitHub repository.
        repo_name (str): The name of the GitHub repository.
        directory_path (str): The path to the directory within the repository.

    Returns:
        list: A list of file URLs for the PDF files in the directory.
    """
    api_url = f"https://api.github.com/repos/{repo_owner}/{repo_name}/contents/{directory_path}"
    headers = {"Accept": "application/vnd.github+json"}
    response = requests.get(api_url, headers=headers)
    response.raise_for_status()

    pdf_files = []
    for file_data in response.json():
        if file_data["type"] == "file" and file_data["name"].endswith(".pdf"):
            download_url = f"https://raw.githubusercontent.com/{repo_owner}/{repo_name}/main/{file_data['path']}"
            pdf_files.append(download_url)
    return pdf_files

# --- Usage ---
repo_owner = "leftgoat"
repo_name = "CloudAdoption"
directory_path = ""  # root directory

pdf_urls = get_github_files(repo_owner, repo_name, directory_path)

# Create a 'docs' directory to save the files
os.makedirs("docs", exist_ok=True)

# Download each PDF
for url in pdf_urls:
    filename = os.path.basename(url)
    print(f"Downloading {filename}...")
    wget.download(url, out=os.path.join("docs", filename))
    print(f" --> Done.")

 --> Done.
 --> Done.
 --> Done.
 --> Done.
 --> Done.
 --> Done.
 --> Done.


##Chunking

Read all the files in the directory

In [ ]:
import os
from langchain.document_loaders import PyPDFLoader

# Assuming PDFs are in a 'docs' folder
pdf_folder_path = 'docs/'

if not os.path.exists(pdf_folder_path):
    print(f"Error: '{pdf_folder_path}' not found. Please upload your PDFs there.")
else:
    # Fix: os.listdir instead of os.listdir
    loaders = [PyPDFLoader(os.path.join(pdf_folder_path, fn)) for fn in os.listdir(pdf_folder_path) if fn.endswith('.pdf')]

    print(f"Found {len(loaders)} PDF documents.")

    docs = []
    for loader in loaders:
        docs.extend(loader.load())

    print(f"Loaded {len(docs)} pages total.")

Found 14 PDF documents.
Loaded 922 pages total.


Split the documents into chunks that are overlapping.

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=25)
splits = text_splitter.split_documents(docs)
print(f"Split into {len(splits)} chunks.")

for i, split in enumerate(splits):
    print(f"Chunk {i + 1}:\n{split.page_content}\n")

Split into 11624 chunks.
Chunk 1:
AWS Whitepaper
AWS Cloud Adoption Framework for 
Artiﬁcial Intelligence, Machine Learning, 
and Generative AI
Copyright © 2025 Amazon Web Services, Inc. and/or its aﬃliates. All rights reserved.

Chunk 2:
AWS Cloud Adoption Framework for Artiﬁcial Intelligence, Machine Learning, 
and Generative AI
AWS Whitepaper
AWS Cloud Adoption Framework for Artiﬁcial Intelligence, Machine

Chunk 3:
Learning, and Generative AI: AWS Whitepaper
Copyright © 2025 Amazon Web Services, Inc. and/or its aﬃliates. All rights reserved.

Chunk 4:
Amazon's trademarks and trade dress may not be used in connection with any product or service 
that is not Amazon's, in any manner that is likely to cause confusion among customers, or in any

Chunk 5:
manner that disparages or discredits Amazon. All other trademarks not owned by Amazon are 
the property of their respective owners, who may or may not be aﬃliated with, connected to, or

Chunk 6:
sponsored by Amazon.

Chunk 7:
AWS Cloud

##Embeddings

In [ ]:
"""##Embedding and Indexing

In this step, we transform the split document chunks into vector embeddings using a pre-trained sentence transformer model (all-MiniLM-L6-v2). These embeddings capture the semantic meaning of the text, enabling efficient similarity-based search. We then store these embeddings in a Chroma vector database, which supports fast retrieval and allows persistence to disk for reuse in future sessions.
"""

from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.retrievers import BM25Retriever, EnsembleRetriever # Import EnsembleRetriever

"""We will use a sentence embedding model"""

embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")
print("Embedding model loaded.")


<ipython-input-5-44c723e9e133>:12: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  war

Embedding model loaded.


##Vector database storage

In [ ]:
persist_directory = "db"  # Specify the directory for persistence
# Create or load the Chroma vector store, enabling persistence to disk.
vectorstore = Chroma.from_documents(
   documents=splits, embedding=embeddings, persist_directory=persist_directory)
vectorstore.persist()
print(f"Vector store created and populated with embeddings, persisted to {persist_directory}.")

Vector store created and populated with embeddings, persisted to db.


<ipython-input-6-c457faf0d730>:5: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectorstore.persist()


# Retrieval & Reranking


Retrieves semantically and keyword-similar chunks using hybrid search,
reranks them, and uses Gemini to answer AWS cloud adoption queries.

**Purpose:** Answers AWS CAF queries using RAG.

**Retrieval:** Combines BM25 keyword search with vector similarity search for hybrid retrieval.

**Reranking:** Uses a BAAI/bge-reranker model to re-prioritize retrieved documents.

**Context:** Selects top reranked documents to create context for the LLM.

**Generation:** Prompts Gemini to produce both Executive and Technical summaries based on the context.

**Argument:**
    *query* (str): The user's question.

**Returns:**
    *str*: Gemini's answer to the question.

In [ ]:
from sentence_transformers import CrossEncoder
from langchain.retrievers import BM25Retriever, EnsembleRetriever

def answer_with_gemini(query):


    # 1. Retrieval of semantically and keyword-similar chunks using hybrid search:
    bm25_retriever = BM25Retriever.from_documents(splits)
    bm25_retriever.k = 5

    vectorstore_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

    ensemble_retriever = EnsembleRetriever(
        retrievers=[bm25_retriever, vectorstore_retriever],
        weights=[0.7, 0.3]
    )

    retrieved_docs = ensemble_retriever.invoke(query)

    # 2. Reranking the retrieved documents using a cross-encoder model
    reranker = CrossEncoder("BAAI/bge-reranker-base")
    doc_pairs = [[query, doc.page_content] for doc in retrieved_docs]
    scores = reranker.predict(doc_pairs)

    # Sort documents by reranker score (descending)
    ranked_docs = [doc for _, doc in sorted(zip(scores, retrieved_docs), key=lambda x: x[0], reverse=True)]

    # Select top 3 reranked documents for generation context
    context = ""
    for doc in ranked_docs[:3]:
        context += doc.page_content + "\n"

    # 3. Construct the prompt for Gemini:
    system_instructions = """
    You are a cloud strategy advisor specializing in AWS Cloud Adoption Framework (CAF).
    Based on the provided {context}, generate two types of output:

    Output 1: Executive Summary
    - Clear and concise explanation suitable for a business executive with limited technical background.
    - Use non-technical, business-aligned language.
    - Focus on value, business outcomes, and high-level strategy.

    Output 2: Technical Advisory Summary
    - Detailed explanation suitable for a cloud architect or IT team.
    - Use precise technical language, AWS terminology, and actionable recommendations.
    - Include architectural or operational considerations where appropriate.

    Provide your responses in the following format and do not repeat the questions.
    """

    prompt = f"""{system_instructions}

    Output 1: Executive Summary

    You are a cloud advisor communicating with senior business stakeholders.
    Read the following AWS context and respond to the questions below:

    {context}

    Questions:
     {query}

    Answer:

    ---

    Output 2: Technical Advisory Summary

    You are a cloud solution architect reviewing cloud adoption documentation.

    Based on the following context:

    {context}

    Please answer:
     {query}

    Answer:
    """

    # 4. Generate the answer using Gemini
    response = client.models.generate_content(
        model=MODEL,
        contents=prompt
    )

    return response.text


### optional if we want to update with other documents

In [ ]:

"""Uploading a new document to the database."""

import os
import sqlite3
from google.colab import files
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.retrievers import BM25Retriever, EnsembleRetriever


def update_db_with_pdf(pdf_path, db_name="my_database.db", persist_directory="db"):
    """Updates the Chroma vector database with a new PDF file."""

    # 1. Upload PDF to the database (if not already present).
    try:
        upload_pdf_to_db(pdf_path, db_name)
        print(f"PDF '{pdf_path}' added to the database.")
    except sqlite3.IntegrityError:
        print(f"PDF '{pdf_path}' already exists in the database. Skipping.")


    # 2. Load and process the new PDF.
    loader = PyPDFLoader(pdf_path)
    new_docs = loader.load()
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=25)
    new_splits = text_splitter.split_documents(new_docs)
    print(f"Loaded and split {len(new_splits)} chunks from the new PDF.")


    # 3. Update the Chroma vector store.
    embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")
    vectorstore = Chroma(persist_directory=persist_directory, embedding_function=embeddings)
    vectorstore.add_documents(new_splits)
    vectorstore.persist()
    print("Vectorstore updated with the new PDF content.")


# Example usage (assuming you have a PDF file uploaded to '/content/docs'):

#new_pdf_path = "/content/docs/your_new_pdf.pdf"  # Replace with actual path
#update_db_with_pdf(new_pdf_path)


# --- Helper functions from original code (slightly modified) ---

def upload_pdf_to_db(pdf_path, db_name="my_database.db"):
    """Uploads PDF content to a SQLite database."""
    with open(pdf_path, "rb") as f:
        pdf_data = f.read()
    conn = sqlite3.connect(db_name)
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS docs (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            filename TEXT UNIQUE,  -- Enforce unique filenames
            data BLOB
        )
    """)
    try:  # Handle potential IntegrityError if the filename exists
        cursor.execute("INSERT INTO docs (filename, data) VALUES (?, ?)", (os.path.basename(pdf_path), pdf_data))
        conn.commit()
    except sqlite3.IntegrityError:
        print(f"File '{os.path.basename(pdf_path)}' already in database. Skipping.")
    conn.close()


# --- Code for file upload and database update ---

uploaded = files.upload()
for filename, data in uploaded.items():
  file_path = os.path.join("/content/docs", filename)
  with open(file_path, "wb") as f:
    f.write(data)
  update_db_with_pdf(file_path)



KeyboardInterrupt: 

#Generation

Google Gemini Set-up (the API key is retrieved automatically from the google Colab)

In [ ]:
!pip install -q -U google-genai  # Install or update google-genai
!pip install -q -U google-generativeai  # Install or update google-generativeai

from google.colab import userdata
from google import genai

# Set your Google API key (ensure it's stored securely)
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
client = genai.Client(api_key=GOOGLE_API_KEY)
MODEL = "gemini-2.0-flash"


User question inputspace, where the output of the model is being displayed

In [ ]:
user_question = "aws cloud adoption overview"  #@param {type:"string"}
answer = answer_with_gemini(user_question)
print(f"Answer: {answer}")

Answer: Output 1: Executive Summary

A Data Perimeter strategy at the Excel maturity level in AWS helps you protect your sensitive information by building strong and intelligent walls around it. Think of it like upgrading from a simple fence to a sophisticated security system for your data. We achieve this by enhancing your existing security foundations with advanced technologies within AWS. This includes improving policies, leveraging Attribute-Based Access Control (ABAC) for granular permissions, and using VPC endpoint policies to control network access to your data. The result is a more robust and automated data protection environment, minimizing risk and ensuring compliance. By implementing this strategy, we move from a basic security posture to a proactive one, demonstrating maturity and improved control over sensitive data assets within your AWS environment.

---

Output 2: Technical Advisory Summary

Achieving the Excel maturity level in the Security Perspective through a Data P

#Evaluation/Testing

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tabulate import tabulate
import google.generativeai as genai
import os

# === Setup Gemini API ===
GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY')
if not GOOGLE_API_KEY:
    try:
        from google.colab import userdata
        GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    except ImportError:
        pass

if not GOOGLE_API_KEY:
    raise ValueError("GOOGLE_API_KEY not found. Please set it as an environment variable or via Colab's userdata.")

genai.configure(api_key=GOOGLE_API_KEY)
model = genai.GenerativeModel('gemini-2.0-flash')

# === Benchmark Questions ===
test_questions = [
    {"type": "Easy", "question": "How do i create an EC2 instance in AWS?"},
    {"type": "Ambiguous", "question": "What are the challenges of moving to the cloud?"},
    {"type": "Domain-Specific", "question": "What is the role of the Business Capability Model in the AWS Cloud adoption?"}
]

# === Safe integer input ===
def safe_int_input(prompt):
    while True:
        try:
            value = int(input(prompt))
            if 1 <= value <= 5:
                return value
            else:
                print("Enter a number between 1 and 5.")
        except ValueError:
            print("Invalid input. Please enter a number.")

# === Generate responses for a question ===
def generate_responses(question_text):
    rag_response = answer_with_gemini(question_text)  # ✅ Your actual system

    base_prompt = f"""
    Output 1: Executive Summary

    You are a cloud advisor communicating with senior business stakeholders.
    Read the following AWS context and respond to the questions below:

    Questions:
     {question_text}

    Answer:

    ---

    Output 2: Technical Advisory Summary

    You are a cloud solution architect reviewing cloud adoption documentation.

    Please answer:
     {question_text}

    Answer:
    """
    try:
        base_response = model.generate_content(base_prompt).candidates[0].content.parts[0].text
    except Exception as e:
        base_response = f"[Error: {e}]"
    return rag_response, base_response

# === Collect ratings ===
def collect_ratings(question_data, rag_response, base_response):
    q = question_data["question"]
    qt = question_data["type"]

    print(f"\n[{qt}] {q}\n")
    print("RAG Response:\n", rag_response)
    print("\nBaseline Response:\n", base_response)

    return {
        "Question Type": qt,
        "Question": q,
        "RAG Relevance": safe_int_input("RAG Relevance (1-5): "),
        "RAG Accuracy": safe_int_input("RAG Accuracy (1-5): "),
        "RAG Completeness": safe_int_input("RAG Completeness (1-5): "),
        "RAG Clarity": safe_int_input("RAG Clarity (1-5): "),
        "Baseline Relevance": safe_int_input("Baseline Relevance (1-5): "),
        "Baseline Accuracy": safe_int_input("Baseline Accuracy (1-5): "),
        "Baseline Completeness": safe_int_input("Baseline Completeness (1-5): "),
        "Baseline Clarity": safe_int_input("Baseline Clarity (1-5): ")
    }

# === Evaluation loop ===
def collect_manual_scores(questions):
    results = []
    for q in questions:
        rag, base = generate_responses(q["question"])
        ratings = collect_ratings(q, rag, base)
        results.append(ratings)
    df = pd.DataFrame(results)
    df["RAG Avg"] = df[["RAG Accuracy", "RAG Completeness", "RAG Clarity"]].mean(axis=1)
    df["Baseline Avg"] = df[["Baseline Accuracy", "Baseline Completeness", "Baseline Clarity"]].mean(axis=1)
    return df

# === Display summary with plots ===
def display_summary(df):
    print("\n📊 Evaluation Summary:")
    print(tabulate(df, headers='keys', tablefmt='fancy_grid'))

    print("\n📈 Average Scores by Type:")
    summary = df.groupby("Question Type")[["RAG Avg", "Baseline Avg"]].mean()
    print(tabulate(summary, headers='keys', tablefmt='grid'))

    # === Bar Plot of Average Scores (All Metrics) ===
    metrics = [
        "RAG Relevance", "RAG Accuracy", "RAG Completeness", "RAG Clarity",
        "Baseline Relevance", "Baseline Accuracy", "Baseline Completeness", "Baseline Clarity"
    ]
    avg_scores = df[metrics].mean().sort_index()

    plt.figure(figsize=(10, 5))
    sns.barplot(x=avg_scores.index, y=avg_scores.values, palette=[
        'skyblue' if 'RAG' in m else 'salmon' for m in avg_scores.index
    ])
    plt.title("Average Evaluation Scores")
    plt.ylabel("Score (1–5)")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    # === Grouped Bar Plot for RAG vs Baseline by Question Type ===
    grouped = df.groupby("Question Type")[["RAG Avg", "Baseline Avg"]].mean().reset_index()
    grouped_melted = grouped.melt(id_vars="Question Type", var_name="System", value_name="Average Score")

    plt.figure(figsize=(10, 6))
    sns.barplot(data=grouped_melted, x="Question Type", y="Average Score", hue="System", palette=["skyblue", "salmon"])
    plt.title("RAG vs Baseline Average Scores by Question Type")
    plt.ylabel("Average Score (1–5)")
    plt.tight_layout()
    plt.show()

# === Entry Point ===
if __name__ == "__main__":
    df_eval = collect_manual_scores(test_questions)
    display_summary(df_eval)
    df_eval.to_csv("rag_evaluation_results.csv", index=False)



[Easy] How do i create an EC2 instance in AWS?

RAG Response:
 Output 1: Executive Summary

The provided context highlights potential security risks associated with how Amazon EC2 instances are being deployed and configured. Specifically, using outdated or unapproved AMIs and overly permissive security groups can expose your organization to vulnerabilities. To create a secure EC2 instance, you need to select an approved and up-to-date "template" (AMI), and then define strict rules about what traffic is allowed in and out (security groups). By following best practices outlined in the AWS Well-Architected Framework, we can ensure your cloud resources are secure and aligned with your business objectives. This improved security posture will protect your data and applications, ultimately reducing risk and enabling you to confidently leverage the cloud.
---

Output 2: Technical Advisory Summary

To create an EC2 instance in AWS, follow these steps, keeping in mind the identified security co